In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import re 
import pandas as pd

# Suppress just SettingWithCopyWarning
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.ChainedAssignmentError)
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
pd.options.mode.chained_assignment = None  # Disable the warning


# Part 1: Getting current routes

Now, we get the current routes for each airport based on their wikipedia article based on a series of scripts developed in the first notebook. Save to a current_routes.csv file. 6


defining key functions

In [ ]:
def get_destinations(iata_source, wiki_name, path_write):
    file = open(path_write, "a") #file to append to
    
    url = f"https://en.wikipedia.org/wiki/{wiki_name}"
    response = requests.get(url)
    
    soup = BeautifulSoup(response.text, 'html.parser')
    #find the related destination table
    # Case-insensitive string match
    heading = soup.find("h2", string=re.compile(r"destination", re.IGNORECASE))
    #check text in heading
    heading_text =  heading.get_text()
    if  "former" in heading_text or "Former" in heading_text: #if either text is found, abort the function. This indicate the airport is no longer in service
        file.close() #close
        return
        
    
    table = heading.find_next("table") 
    while ('wikitable' not in table.get("class")): #find the next table matching a predictable class, if one has not been found
        table = table.find_next("table") 
    rows = table.find_all("tr")

    
    for i in range(1,len(rows)): #exclude the first row
        row = rows[i]
        # Extract all cells (td or th)
        cols = row.find_all(["td", "th"])
        # Write the row text content to CSV
        #first column is the airline
        airline = cols[0].get_text(strip=True)
        #get the list of destinations in the 2nd  
        destinations = cols[1]
        isSeasonal = 0 #iterate over subcomponents (seasonal always comes last, so set is seasonal to be false for now)
        for child in destinations.children: 
            #anchor components are the only destinations
            if (child.name == "a"):
                dest_name = child.get('title') #the title is the official wikipedia airport name (without _ in place of spaces)
                dest_name = dest_name.replace(" ", "_") 
                output = f"\"{iata_source}\",\"{wiki_name}\",\"{dest_name}\",\"{airline}\",\"{isSeasonal}\"\n" #final output to append to the file
                file.write(output)#write file
            elif ((child.name == "b") and (child.text == "Seasonal:")):
                isSeasonal = 1 #get seasonal to be 1 for future destinations
    file.close() #close
    return

## start

Start the file to store modern route data

In [ ]:
f = open("./data/current_routes.csv", "w")
f.write("iata_source,starting_wiki_name,dest_wikipedia_name,airline,isSeasonal\n") 
f.close() #add column names

get iata code, wikiname lists

In [ ]:
data = pd.read_csv("./data/top_airports_basic_data.csv")
print(len(data))
data.head(n=1)
codes_list = data["iata"]
names_list = data["wiki_name"]
file_append_path = "./data/current_routes.csv"

iterrate through airports 0 to 50

In [ ]:

for i in range(0,50):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")


iterrate through airports 50 to 100

In [ ]:
for i in range(50,100):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")

iterate through airports 100 to 150

In [ ]:
for i in range(100, 150):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")

iterate through airports 150 to 200

In [ ]:
for i in range(150,200):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")

iterate through airports 200 to 250

In [ ]:
for i in range(200,250):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")

iterate through airports 250 to 300

In [ ]:
for i in range(250,300):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")

iterate through airports 300 to 350

In [ ]:
for i in range(300,350):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")

iterate through the rest of the airports

In [ ]:
for i in range(350, len(codes_list)):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")

### fixing failed airports and updating airport data

A few airports were failing, due to outdated data, mainly due to some airports in cities like Qingdao closing, along with faulty links from the original query for some smaller airports. The updated wikipedia names were found for airports in cities where the main airport had changed (Berlin Tegel, Istanbul Ataturk, for example), with the following functions ran to add data like below.

A new table of airports based on the routes table here will be created to used be as a reference, with the same information obtained by wikipedia api

We find the missing airports like below:

In [ ]:
routes = pd.read_csv("./data/current_routes.csv")
unique_airports_in_routes = set(routes["iata_source"].unique())
#find missing airports
missing = set(codes_list) - unique_airports_in_routes
for m in missing:
    print("missing airports", m)

adding some  missing entries (with the exception of executive/closed airports or those without destinations on wikipedia)

Some new airports have destinations added in the case that the old iata code was replace

In [ ]:
get_destinations("LDU","Lahad_Datu_Airport",file_append_path)

In [ ]:
get_destinations("CGY","Laguindingan_Airport",file_append_path)

In [ ]:
#new Saratov airport -replacing RTW, removed RTW entries
get_destinations("GSV","Saratov_Gagarin_Airport",file_append_path)

In [ ]:
#new Berlin airport- replacing TXL, SXF 
get_destinations("BER", "Berlin_Brandenburg_Airport",file_append_path)

In [ ]:
get_destinations("UTH", "Udon_Thani_International_Airport",file_append_path)

In [ ]:

get_destinations("VAS", "Sivas_Airport",file_append_path)

In [ ]:
get_destinations("TER", "Lajes_Airport",file_append_path)

In [ ]:

get_destinations("SYZ", "Shiraz_Shahid_Dastgheib_International_Airport",file_append_path)

In [ ]:

get_destinations("SNO", "Sakon_Nakhon_Airport",file_append_path)

In [ ]:
get_destinations("TTE", "Sultan_Babullah_Airport",file_append_path)

In [ ]:

get_destinations("NST","Nakhon_Si_Thammarat_Airport",file_append_path)

In [ ]:
obt = ("SOC", "Adisoemarmo_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
#new Rajkot airport, replacing RAJ
obt = ("HSR", "Rajkot_International_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
#replacing TAG, old TAG entries in routes removed
obt = ("TAG", "Bohol–Panglao_International_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
# New samarinda,indonesia airport, replacing SRI, old entries revmoed
obt = ("AAP", "Aji_Pangeran_Tumenggung_Pranoto_International_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("DIN", "Dien_Bien_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("UBJ", "Yamaguchi_Ube_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("KUV", "Gunsan_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("HMA", "Khanty-Mansiysk Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("WGA", "Wagga_Wagga_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("GRV", "Kadyrov_Grozny_International_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("TAO", "Qingdao_Jiaodong_International_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("HRB", "Harbin_Taiping_International_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("YNT", "Yantai_Penglai_International_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("ZAZ", "Zaragoza_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("THS", "Sukhothai_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
#new Murcia airport, replacing MJV
obt = ("RMU", "Región_de_Murcia_International_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("MSJ", "Misawa_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("ISG", "New_Ishigaki_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("TIM", "Mozes_Kilangin_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("UBP", "Ubon_Ratchathani_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("HFE", "Hefei_Xinqiao_International_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("MLX", "Malatya_Erhaç_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("REU", "Reus_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("YKS", "Platon_Oyunsky_Yakutsk_International_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
#new istanbul airport
obt = ("IST", "Istanbul_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("BMV", "Buon_Ma_Thuot_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("ROV", "Platov_International_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("TRZ", "Tiruchirappalli_International_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("RBR", "Rio_Branco_International_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("KOP", "Nakhon_Phanom_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("JOG", "Adisutjipto_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("UUS", "Yuzhno-Sakhalinsk_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("NSN", "Nelson_Airport_(New_Zealand)")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("NUX", "Novy_Urengoy_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("INC", "Incheon_International_Airport")
get_destinations(obt[0],obt[1],file_append_path)

In [ ]:
obt = ("PMW", "Palmas_Airport")
get_destinations(obt[0],obt[1],file_append_path)

### checking missing airports again, verifying none have further passengers

In [ ]:
routes = pd.read_csv("./data/current_routes.csv")
unique_airports_in_routes = set(routes["iata_source"].unique())
#find missing airports
missing = set(codes_list) - unique_airports_in_routes
for m in missing:
    print("missing airports", m)

dropping duplicate data

In [ ]:
routes= routes.drop_duplicates() #drop rows where all values are the same (should not be any)
print(len(routes))
routes.to_csv("./data/current_routes.csv")

### fixing airline names with [] quotations, removing them

In [ ]:
new_airlines_list = []
old_airlines_list = routes["airline"]
for airline in old_airlines_list:
    airline = airline.split("[")[0] #remove quotation
    new_airlines_list.append(airline)
routes["airline"] = new_airlines_list
routes.head()

In [ ]:
routes.to_csv("./data/current_routes.csv")

### Some basic analysis on our preliminary current routes data

In [ ]:
print("number of unique airlines:", routes["airline"].nunique())
print("number of unique destinations:", routes["dest_wikipedia_name"].nunique())

# Part 2: Creating a database of current airports served

We now use wikipedia to create a table of detailed information for all the destination airport names in the current routes.csv, which can be done easily since we know the wikipedia names, using the GlobalAirportDatabase data downloaded (some detailed information may be outdated), but we only want a general location and accurate coordinates to within 30mi (to estimate flight paths), which is not too restrictive.

In [ ]:
routes = pd.read_csv("./data/current_routes.csv")
destinations_wiki_names = routes["dest_wikipedia_name"].unique()

create the csv file storing this information

In [ ]:
f = open("./data/current_served_airports.csv", "w")
f.write("IATA,wiki_name,city,country,latitude,longitude\n") 
f.close() #add column names

link to global airport database, at https://www.partow.net/miscellaneous/airportdatabase/  and guide. Loading in the database so the city 

In [ ]:
db_names=["ICAO","IATA","Airport_name", "City", "Country", 
          "lat_deg","lat_min","lat_sec","lat_dir",
          "long_deg","long_min","long_sec","long_dir",
          "Altitude", "lat_dec", "long_dec"]#names given on the website in order
airport_db = pd.read_csv("./data/GlobalAirportDatabase.txt", delimiter=":",names=db_names)
airport_db.head()

create a function to find the coordinates, Country, City of a airport given an iata code

In [ ]:
def getdetails(iata, airport_db):
    try:
        matching = airport_db[airport_db["IATA"]==iata].iloc[0]
        return {"city":matching["City"], "country":matching["Country"],"latitude":matching["lat_dec"], "longitude":matching["long_dec"]}
    except:
        return {"city":"", "country":"","latitude":"", "longitude":""}
getdetails("JFK", airport_db)

creating getting raw text from wikipedia to get the iata code of any airport


In [ ]:
def getIataFromWikiName(wiki_name):
    url = f"https://en.wikipedia.org/w/index.php?title={wiki_name}&action=raw"
    response = requests.get(url)
    #find the text between
    text = response.text
    
    try:
        match = re.findall(r'IATA.*?=', text)[0] #find the first indstance
        start = text.find(match) #find the starting index, by matching the re pattern iata*=
        start += len(match) #do not include iata
        
        end = text.find("\n", start) #starting from the end, find the starting index
        iata_code = text[start:end]
        iata_code = iata_code.split("<")[0]#get rid of ref tags
        iata_code = iata_code.strip()
        return iata_code
    except:
        #check for redirect
        check_text = text.split("[[")[0] #get section between [[
        check_text = check_text.lower()
        if ("redirect" in check_text):
            #get the text in between [[]]
            redirect = text.split("]]")[0]
            redirect = redirect.split("[[")[1]
            redirect = redirect.replace(" ", "_") #replace spaces
            print("redirect found:", redirect)
            return getIataFromWikiName(redirect)
        return "x"

Testing for redirects and modifying functions to conduct directs (for sample Dallas/Fort Worth International Airport) should map to the same iata code as  Dallas_Fort_Worth_International_Airport (wiki names for the same airport are not necessarily unique)

In [ ]:
getIataFromWikiName("Dallas/Fort Worth International Airport")

In [ ]:
getIataFromWikiName("Genoa_Airport")

In [ ]:
getIataFromWikiName("Victoria_Falls_International_Airport")

attempting to get details for all ~ 4000 airports

In [ ]:

i = 0
for dest_wiki in destinations_wiki_names:
    try:
        f = open("./data/current_served_airports.csv", "a")
        print("index is:",i)
        iata_code = getIataFromWikiName(dest_wiki)
        matching = getdetails(iata_code, airport_db) #details object 
        city =  matching["city"]
        country = matching["country"]
        lat = matching["latitude"]
        long = matching["longitude"]
        output = f"\"{iata_code}\",\"{dest_wiki}\",\"{city}\",\"{country}\",\"{lat}\",\"{long}\"\n" #write IATA,wiki_name,country,city,latitude,longitude
        f.write(output)
        f.close()    
    except:
        output = f"\"{iata_code}\",\"{dest_wiki}\",\"\",\"\",\"\",\"\"\n" #write IATA,wiki_name,country,city,latitude,longitude
        f.write(output)
        f.close()
        continue
    i += 1

### checking invalid iata code, iterating through csv

In [ ]:
airports = pd.read_csv("./data/current_served_airports.csv", on_bad_lines="skip")
print(len(airports))
airports.head(n=1)

In [ ]:
def checkiata(airports):
    for index, row in airports.iterrows():
        iata = row["IATA"]
        name = row["wiki_name"]
        try:
            if len(iata) != 3:
                print(f"faulty iata:{iata}| for name: {name}") 
        except:
            print(f"faulty iata:{iata}| for name: {name}") 
    return

In [ ]:
checkiata(airports)

notice that Basel airport has 3 iata codes. This is valid. However, we dropping "nan" value

In [ ]:
drop = []
for index, row in airports.iterrows():
    iata = row["IATA"]
    iata = str(iata)
    if iata == "nan":
        drop.append(index)

In [ ]:
print(drop)

In [ ]:
airports = airports.drop(index=drop)
airports = airports.reset_index(drop=True) #reindex (dropping old index)

try to rerun the updated get Iata program on invalid iata 

In [ ]:
for index, row in airports.iterrows():
    iata = str(row["IATA"])
    name = row["wiki_name"]
    try:
        if len(iata) != 3:
            iata_code = getIataFromWikiName(name)
            matching = getdetails(iata_code, airport_db) #details object 
            airports["IATA"][index] = iata_code
            airports["city"][index] = matching["city"]
            airports["country"][index] = matching["country"]
            airports["latitude"][index] = matching["latitude"]
            airports["longitude"][index] = matching["longitude"]
    except:
        continue
    

check iata code:

In [ ]:
checkiata(airports)

do a final drop, exclude Basel case, and 2 other special cases found

In [ ]:
drop = []
for index, row in airports.iterrows():
    iata = str(row["IATA"])
    name = row["wiki_name"]
    if len(iata) != 3:
        if "BSL, MLH, EAP" != iata and "OIBH" != iata:
            print("dropped iata code:", iata)
            drop.append(index)
        if "IKU" in iata :
            iata_code = "IKU"
            matching = getdetails(iata_code, airport_db) #details object 
            airports["IATA"][index] = iata_code
            airports["city"][index] = matching["city"]
            airports["country"][index] = matching["country"]
            airports["latitude"][index] = matching["latitude"]
            airports["longitude"][index] = matching["longitude"]
        if "''TRT''" in iata :
            iata_code = "TRT"
            matching = getdetails(iata_code, airport_db) #details object 
            airports["IATA"][index] = iata_code
            airports["city"][index] = matching["city"]
            airports["country"][index] = matching["country"]
            airports["latitude"][index] = matching["latitude"]
            airports["longitude"][index] = matching["longitude"]
    


In [ ]:
drop

In [ ]:
airports = airports.drop(index=drop)
airports = airports.reset_index(drop=True) #reindex (dropping old index)

save data

In [ ]:
len(airports)
#
airports.to_csv("./data/current_served_airports.csv")

### now, add iata codes when possible to the current routes database for the destination airports

In [ ]:
routes_data = pd.read_csv("./data/current_routes.csv")
airports = pd.read_csv("./data/current_served_airports.csv")

In [ ]:
routes_data.head(n=1)

In [ ]:
airports.head(n=1)

In [ ]:
#add iata_dest_source
routes_data["iata_dest"] = None 
for index, rows in routes_data.iterrows():
    print("current row:", index)
    dest_wiki = rows["dest_wikipedia_name"]
    try:  #try to match to a iata code
        match = airports[airports["wiki_name"]==dest_wiki].iloc[0]
        match = match["IATA"]
        routes_data["iata_dest"][index] = match
    except:
        routes_data["iata_dest"][index] = None

save new data

In [ ]:
routes_data.to_csv("./data/current_routes.csv", index=False)

In [ ]:
routes_data.head(n=1)

find routes without any valid iata (null)

In [ ]:
none_dest_Data = routes_data[routes_data["iata_dest"].isnull()]
print(len(none_dest_Data))

In [ ]:
none_dest_Data.head()

Therefore, it turns out the percentage of routes, without a destination IATA is low (282/76500) ~ 0.36 %

# Part 3: Narrowing down to create database of airports in the database listed as origins

### populating data 

We now narrow down to airports where there is an origin flights. In this case, we require all rows to have latitude, longitude for the purpose of distance calculations later.

In [ ]:
routes = pd.read_csv("./data/current_routes.csv", encoding='utf-8')
iata_sources = routes["iata_source"].unique()
print(len(iata_sources))
routes.head(n=1)

In [ ]:
source_airport_link = "./data/current_source_airports.csv"
data = [
    ["IATA","wiki_name","city","country","latitude","longitude"]
]


load reference data of the current served airports. It must be the case that the served airports is a superset of those origin ones

In [ ]:
ref_data =pd.read_csv("./data/current_served_airports.csv", encoding='utf-8')
ref_data.head(n=1)

Get all details of these ~ 980 airports

In [ ]:
i = 0 
for iata_source in iata_sources:
    try:
        print("index is:",i)
        #match based on reference data 
        matching = ref_data[ref_data["IATA"]==iata_source].iloc[0]
        wiki_name =  str(matching["wiki_name"])
        city =  str(matching["city"])
        country = str(matching["country"])
        lat = str(matching["latitude"])
        long = str(matching["longitude"])
        data.append([iata_source, wiki_name, city,country,lat,long])

    except:
        data.append([iata_source, wiki_name, "","","",""])
        continue
    i += 1
with open(source_airport_link, 'w', encoding='utf-8', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerows(data)

### Repairing data 

Getting rows that are empty for the iata code

In [ ]:
source_airport_link = "./data/current_source_airports.csv"
current_airports =pd.read_csv(source_airport_link, encoding='utf-8')
print(len(current_airports))
current_airports.head(n=1)

In [ ]:
len(current_airports[current_airports["IATA"].isnull()])

Checking rows that are empty for details like latitude

In [ ]:
empty_rows = current_airports[current_airports["latitude"].isnull()]
print(len(empty_rows))

In [ ]:
empty_rows.head(n=196)

Get list of wikipedia names that are empty, convert to set for quick checking

In [ ]:
wiki_names = set(empty_rows["wiki_name"])
print(len(wiki_names))

Loading function to get city, country, latitude, and longitude from wikipedia

In [ ]:
def getRow(text, key): #help function to text a key from = of the first rpws
    try:
        regex = f"{key}"+'.*?='
        match = re.findall(rf'{regex}', text)[0] #find the first indstance
        
        start = text.find(match) #find the starting index, by matching the re pattern iata*=
        start += len(match) #do not include iata
        
        end = text.find("\n", start) #starting from the end, find the starting index
        code = text[start:end]
        code = code.split("<")[0]#get rid of ref tags
        code = code.strip()
        return code
    except:
        #check for redirect
        check_text = text.split("[[")[0] #get section between [[
        check_text = check_text.lower()
        if ("redirect" in check_text):
            #get the text in between [[]]
            redirect = text.split("]]")[0]
            redirect = redirect.split("[[")[1]
            redirect = redirect.replace(" ", "_") #replace spaces
            #get text from redirect
            url = f"https://en.wikipedia.org/w/index.php?title={wiki_name}&action=raw"
            response = requests.get(url)
            #find the text between
            text = response.text
            return getRow(text, key)
        return "" #return empty string if nothing is found

#function to convert DMS coordinates on wiki to decimal ones
def dms_to_decimal(degrees, minutes, seconds, direction):
    dd = float(degrees) + float(minutes) / 60 + float(seconds) / 3600
    if direction.upper() in ['S', 'W']:
        dd *= -1
    return str(dd)



def getDetailsFromWikiName(wiki_name):
    url = f"https://en.wikipedia.org/w/index.php?title={wiki_name}&action=raw"
    response = requests.get(url)
    #find the text between
    text = response.text
    #intialize as empty strings
    city = ""
    country = ""
    lat = ""
    long = ""
    try:
        #process city data
        city = getRow(text,"city-served")#city
        if "[[" in city:
            city = city.split("[[")[1]
        city = city.split("]]")[0]
        # process latitude, longtitude data
        coor = getRow(text,"coordinates")
        coor = coor.split("}}")[0]
        coor = coor.split("{{")[1]
        coor = coor.split("|")
        lat = dms_to_decimal(coor[1],coor[2],coor[3],coor[4]) #convert using function
        long = dms_to_decimal(coor[5],coor[6],coor[7],coor[8])
        #get country data, using wikipedia api using city
        url2 = f"https://en.wikipedia.org/w/index.php?title={city}&action=raw"
        response2 = requests.get(url2)
        #find the text between
        text2 = response2.text
        country = getRow(text2,"subdivision_name")
        #depending on the enclosing symbol
        if "[" in country:
            country = country.split("[[")[1]#get between [[]]
            country = country.split("]]")[0]
        elif "{" in country:
            #get between (())
            country = country.split("{{")[1]#get between [[]]
            country = country.split("}}")[0]
        if "|" in country: #now, check for |
            tlist = country.split("|")
            country = tlist[len(tlist)-1]
        return {"city":city, "country":country, "latitude":lat, "longitude":long}
    except:
        #check for redirect
        check_text = text.split("[[")[0] #get section between [[
        check_text = check_text.lower()
        if ("redirect" in check_text):
            #get the text in between [[]]
            redirect = text.split("]]")[0]
            redirect = redirect.split("[[")[1]
            redirect = redirect.replace(" ", "_") #replace spaces
            return getDetailsFromWikiName(redirect)
        return {"city":city, "country":country, "latitude":lat, "longitude":long}
        
        

Testing

In [ ]:
getDetailsFromWikiName("Shanghai_Pudong_International_Airport")

In [ ]:
getDetailsFromWikiName("Incheon_International_Airport")

In [ ]:

data = getDetailsFromWikiName("Mashhad_International_Airport")
print(data)
print(data["city"])

Try to fill out missing cells. do this with different index ranges at a time to avoid overloading the wikiapi

In [ ]:
def repair(start, end, current_airports):
    f = open("temp.txt", "w", encoding='utf-8')#open a temp progress txt
    for index, row in current_airports.iterrows():
        if index in range(start, end):
            print(f"current index: {index}")
            wiki_name = row["wiki_name"]
            if wiki_name in wiki_names: #if this is in the empty list
                try:
                    #write 
                    f = open("temp.txt", "a", encoding='utf-8')#open a temp progress txt for appending
                    f.write(f"current wiki name: {wiki_name}\n")
                    data = getDetailsFromWikiName(wiki_name)
                    f.write(f"data found for missing rows at index:{index}\n")
                    f.close()
                    current_airports.at[index, "city"] = data["city"]
                    current_airports.at[index, "country"] = data["country"]
                    current_airports.at[index, "latitude"] = float(data["latitude"])
                    current_airports.at[index, "longitude"] = float(data["longitude"])
                except:
                    current_airports.at[index, "city"] = ""
                    current_airports.at[index, "country"] = ""
                    current_airports.at[index, "latitude"] = ""
                    current_airports.at[index, "longitude"] = ""
            else:
                continue
        else:
            continue
    return current_airports #return the modified data
        

In [ ]:
#conducting for indices 0-187
current_airports = repair(0, 187, current_airports)

In [ ]:
#conducting for indices 187-300
current_airports = repair(187,300, current_airports)

In [ ]:
#conducting for indices 300-500
current_airports = repair(300,500, current_airports)

In [ ]:
#conducting for indices 500-700
current_airports = repair(500,700, current_airports)

In [ ]:
#conducting for indices 700-
current_airports = repair(700, len(current_airports), current_airports)

In [ ]:
# Count rows where any column is null
for col in current_airports.columns:
    print(f"current column {col}")
    num_rows_with_nulls = current_airports[current_airports[col]== ""]
    print(f"number of empty rows in the {col} column:", len(num_rows_with_nulls))

In [ ]:
#save .to_csv()
current_airports.to_csv("./data/current_source_airports.csv", index=False, encoding='utf-8') #save without index

In [ ]:
current_airports.head(n=10)

### manual repair

Final part: inspecting and manually repairing empty rows, first repairing those without a latitude/longitude, as this is the most important part (needed to calculate distances later for graph algorithm)

In [ ]:

current_airports= pd.read_csv("./data/current_source_airports.csv", encoding='utf-8')
current_airports.head(n=7)

In [ ]:
# Count rows where any column is null
for col in current_airports.columns:
    print(f"current column {col}")
    num_rows_with_nulls = current_airports[current_airports[col].isnull()]
    print(f"number of empty rows in the {col} column:", len(num_rows_with_nulls))

In [ ]:
rows_with_nulls = current_airports[current_airports["latitude"].isnull()]
print(rows_with_nulls)

In [ ]:
current_airports.loc[70] = ["KIX", "Kansai_International_Airport", "Osaka", "Japan",34.4272,135.244]
current_airports.loc[392] = ["KUF", "Kurumoch_International_Airport", "Samara", "Russia",53.501667, 50.155]
current_airports.loc[623] = ["GNY", "Şanlıurfa_GAP_Airport", "Şanlıurfa", "Turkey",37.45, 38.9]

current_airports.loc[744] = ["MWX", "Muan_International_Airport", "Muan", "South Korea",34.991406, 126.382814]
current_airports.loc[790] = ["CSV", "Cheboksary_International_Airport", "Cheboksary", "Russia",56.0903, 47.3472]

current_airports.loc[818] = ["IGT", "Magas_Airport", "Magas", "Russia",43.3193, 45.013]
current_airports.loc[947] = ["TAG", "Bohol–Panglao_International_Airport", "Bohol", "Philippines",9.566667, 123.775]

In [ ]:
current_airports.to_csv("./data/current_source_airports.csv", index=False, encoding='utf-8') #save without index

last step : standardize uppercase/lowercase format in the csv for city  and country

In [ ]:
for index, row in current_airports.iterrows():
    city = row["city"]
    country = row["country"]
    print("current index", index)
    try: 
        #all lower case by spaces 
        city_list = city.split(" ")
        new_city_name = "" #new city name
        for part in city_list: 
            part = part.lower()
            part = part[0].upper() + part[1:] #string immutable, must be done this way
            new_city_name += part + " "
        #all lower case by spaces 
        country_list = country.split(" ")
        new_country_name = "" #new city name
        for part in country_list: 
            part = part.lower()
            part = part[0].upper() + part[1:]
            new_country_name += part + " "
        #trim the extra space
        new_city_name = new_city_name.strip()
        new_country_name = new_country_name.strip()
        current_airports.at[index, "city"] = new_city_name
        current_airports.at[index, "country"] = new_country_name
    except:
        continue
        

In [ ]:
current_airports.to_csv("./data/current_source_airports.csv", index=False, encoding='utf-8') #save without index

After saving and fixing the capitalization of Dallas-Fort Worth, we are done